# Import

In [1]:
import torch
import Trainer_CLT as Trainer
from transformers import AutoTokenizer
import BertEnchoder_CLT as BertEnchoder
import pickle

# Tokenizxer

In [2]:
tzer = AutoTokenizer.from_pretrained("Level Wise Training/CseBlueNlpTokeniser")

In [3]:
tzer

ElectraTokenizerFast(name_or_path='Level Wise Training/CseBlueNlpTokeniser', vocab_size=32000, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

# Data Loader

In [4]:
data = torch.load("Level Wise Training/Sorted_Entropy_Data_50.pt")
train = data['Train']
test = data['Test']

/tmp/ipykernel_33943/3201692631.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load("Level Wise Training/Sorted_Entropy_Data_50.pt")


In [5]:
print(train.keys(),test.keys())

dict_keys(['input_ids', 'attention_mask']) dict_keys(['input_ids', 'attention_mask'])


In [6]:
print(train['input_ids'][:20])
print(train['attention_mask'][:20])

tensor([[    2,     1,    96,  ...,     1,    96,     3],
        [    2,     1,    96,  ...,     1,    96,     3],
        [    2,     1,    96,  ...,     1,    96,     3],
        ...,
        [    2, 19484,  6732,  ...,     0,     0,     0],
        [    2,  1594,   800,  ...,     0,     0,     0],
        [    2,  3527, 31517,  ...,     0,     0,     0]], dtype=torch.int32)
tensor([[False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        ...,
        [False, False, False,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True]])


In [7]:
print(test['input_ids'][:20])
print(test['attention_mask'][:20])

tensor([[    2,   830,  3300,  ...,     0,     0,     0],
        [    2, 27581,  2926,  ...,  9177,  1031,     3],
        [    2,     1,  4451,  ...,     0,     0,     0],
        ...,
        [    2,   819, 21391,  ..., 23415,   955,     3],
        [    2, 11490, 23618,  ...,     0,     0,     0],
        [    2,   205, 25098,  ...,     0,     0,     0]], dtype=torch.int32)
tensor([[False, False, False,  ...,  True,  True,  True],
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ...,  True,  True,  True],
        ...,
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True]])


In [8]:
length_train = len(train['input_ids'])
length_test = len(test['input_ids'])

In [9]:
train_data = torch.utils.data.TensorDataset(train['input_ids'][:int(length_train * 1)], train['attention_mask'][:int(length_train * 1)])
test_data = torch.utils.data.TensorDataset(test['input_ids'][:int(length_test * 1)], test['attention_mask'][:int(length_test * 1)])

In [10]:
train_data_loder = torch.utils.data.DataLoader(train_data, batch_size=96, shuffle=True, pin_memory=True,persistent_workers=True,num_workers=8,prefetch_factor=2)
test_data_loder = torch.utils.data.DataLoader(test_data, batch_size=96, shuffle=False, pin_memory=True,persistent_workers=True,num_workers=8,prefetch_factor=2)

In [11]:
del train_data
del test_data

In [12]:
len(train_data_loder),len(test_data_loder)

(13993, 3499)

# Model And Training Para Meteres

In [13]:
epochs = 7
back_step_every = 8
steps = int(epochs * int((len(train_data_loder) / back_step_every) + int((len(train_data_loder) % back_step_every) != 0)))
w_steps = max(1, int(steps * 0.01))
c_steps1 = int(max(1,(steps)*5/7  - w_steps))
c_steps2 = int(max(1,(steps - w_steps - c_steps1)))

In [14]:
model_name = "Model_Post_Norm_CLT_1"
model = BertEnchoder.EncoderOnly(vocabSize=tzer.vocab_size,embedDim=288,numHeads=6,
                                 numLayers=18,numPosEmbeading=128,padIdx=0,
                                 Same_Weights_Out_EMbed= True)
device = torch.device('cuda')
model = model.to(device)
print(model)

EncoderOnly(
  (Embead): BERTmbeadings(
    (Embead): Embedding(32000, 288, padding_idx=0)
    (PosEmbead): Embedding(128, 288)
    (Norm): LayerNorm((288,), eps=1e-05, elementwise_affine=True)
  )
  (Blocks): ModuleList(
    (0-17): 18 x EncoderBLock(
      (Attantion): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=288, out_features=288, bias=True)
      )
      (MLP): Sequential(
        (0): Linear(in_features=288, out_features=1152, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=1152, out_features=288, bias=True)
      )
      (Norm1): LayerNorm((288,), eps=1e-05, elementwise_affine=True)
      (Norm2): LayerNorm((288,), eps=1e-05, elementwise_affine=True)
      (Drop1): Dropout(p=0.1, inplace=True)
      (Drop2): Dropout(p=0.1, inplace=True)
    )
  )
  (MLM): MaskedLangModeling(
    (MLP): Sequential(
      (0): Linear(in_features=288, out_features=288, bias=True)
      (1): GELU(approximate='none')
      (2)

# Train Functions and Classes \_\_init__

In [15]:
loss = Trainer.Loss_Class(torch.nn.CrossEntropyLoss())

In [16]:
start_lr = 3e-3
cosin_1_end = 6e-4
cosin_2_strat = 1e-3
cosin_2_end = 5e-5
start_factor = 5e-1

In [17]:
optim = Trainer.Optimizer_Class(
                    torch.optim.AdamW(params=model.parameters(),betas=(0.9,0.95),lr=start_lr,weight_decay=1e-3,fused=True),
                    step_every=back_step_every)

In [18]:

Cosine1 = torch.optim.lr_scheduler.CosineAnnealingLR(optim.optim,T_max=c_steps1,eta_min= cosin_1_end)
for param_group in optim.optim.param_groups:
    param_group['initial_lr'] = cosin_2_strat
Cosine2 = torch.optim.lr_scheduler.CosineAnnealingLR(optim.optim,T_max=c_steps2,eta_min= cosin_2_end)
for param_group in optim.optim.param_groups:
    param_group['initial_lr'] = start_lr
warmup = torch.optim.lr_scheduler.LinearLR(optim.optim,start_factor= start_factor, total_iters=w_steps)

scheduler = torch.optim.lr_scheduler.SequentialLR(optim.optim,[warmup, Cosine1, Cosine2,],
                                                  milestones=[w_steps, w_steps + c_steps1])

scheduler = Trainer.Scheduler_Class_CLT(scheduler,gamma_max=1.2, gamma_min=0.8,
                                        gama_scheduler_steps=int(steps * (6/7)),
                                        step_every=back_step_every)

In [19]:
from importlib import reload

In [20]:
reload(Trainer)

<module 'Trainer_CLT' from '/home/bulu/Desktop/Level Wise Training/ContinualLevelWiseTraining/Trainer_CLT.py'>

In [21]:
ff = Trainer.Forward_Class(mask_id=4, change_prob=.2, p_rand = .1, p_unchanged= .1)

In [22]:
Trainer_Class = Trainer.Trainer(model=model,accuracy_func=Trainer.accuracy_func,
                                forward_func=ff,loss_class=loss,optimizer_class=optim,
                                save_func=Trainer.save2,scheduler_class=scheduler,
                                user_metrics_class=Trainer.User_Metrices())

# Training and Results

In [23]:
# model.load_state_dict(weight['state_dict'])

In [24]:
accu_and_loss = Trainer_Class.train(epochs=epochs,train_data=train_data_loder,
                                    disable_pbar = False,test_data=test_data_loder,enable_bf16=True,
                                    enable_validation_bf16=True,save_name = model_name)

Epoch Number: 1 / 7
Training


100%|██████████| 13993/13993 [1:03:08<00:00,  3.69it/s, MLM_Loss=5.805958, MLM_Accu=0.243851, Overall_Accu=0.801719]


Validation


100%|██████████| 3499/3499 [05:05<00:00, 11.45it/s, MLM_Loss=4.703010, MLM_Accu=0.317085, Overall_Accu=0.837830]


torch.Size([864, 288]) 45.34037399291992
torch.Size([864]) 3.0301320552825928
torch.Size([288, 288]) 17.431434631347656
torch.Size([288]) 1.684975266456604
torch.Size([1152, 288]) 54.73843765258789
torch.Size([1152]) 5.8582916259765625
torch.Size([288, 1152]) 42.93471908569336
torch.Size([288]) 1.2611898183822632
torch.Size([288]) 15.503544807434082
torch.Size([288]) 3.677757740020752
torch.Size([288]) 14.764102935791016
torch.Size([288]) 1.0807981491088867
_______________
torch.Size([864, 288]) 45.439697265625
torch.Size([864]) 2.085190534591675
torch.Size([288, 288]) 23.61905860900879
torch.Size([288]) 1.2187703847885132
torch.Size([1152, 288]) 59.223514556884766
torch.Size([1152]) 4.624914646148682
torch.Size([288, 1152]) 53.27924728393555
torch.Size([288]) 0.9612488746643066
torch.Size([288]) 15.89773178100586
torch.Size([288]) 2.0093672275543213
torch.Size([288]) 15.18895435333252
torch.Size([288]) 0.7460283041000366
_______________
torch.Size([864, 288]) 44.46894836425781
torch.S

...

torch.Size([864, 288]) 64.86528015136719
torch.Size([864]) 4.9679131507873535
torch.Size([288, 288]) 19.152965545654297
torch.Size([288]) 2.2652108669281006
torch.Size([1152, 288]) 74.46807098388672
torch.Size([1152]) 8.78512191772461
torch.Size([288, 1152]) 63.220436096191406
torch.Size([288]) 1.5531182289123535
torch.Size([288]) 12.241386413574219
torch.Size([288]) 8.079236030578613
torch.Size([288]) 11.069514274597168
torch.Size([288]) 2.0133168697357178
_______________
torch.Size([864, 288]) 60.97298812866211
torch.Size([864]) 3.6604559421539307
torch.Size([288, 288]) 20.129676818847656
torch.Size([288]) 2.065106153488159
torch.Size([1152, 288]) 81.65272521972656
torch.Size([1152]) 6.823332786560059
torch.Size([288, 1152]) 70.2977066040039
torch.Size([288]) 2.1069414615631104
torch.Size([288]) 13.95315170288086
torch.Size([288]) 4.886287212371826
torch.Size([288]) 12.78194522857666
torch.Size([288]) 1.615515947341919
_______________
torch.Size([864, 288]) 61.00141143798828
torch.Si

100%|██████████| 13993/13993 [1:03:28<00:00,  3.67it/s, MLM_Loss=3.564468, MLM_Accu=0.417698, Overall_Accu=0.861935]


Validation


100%|██████████| 3499/3499 [05:13<00:00, 11.17it/s, MLM_Loss=3.422876, MLM_Accu=0.435803, Overall_Accu=0.871887]


torch.Size([864, 288]) 64.91722106933594
torch.Size([864]) 4.974847316741943
torch.Size([288, 288]) 19.018817901611328
torch.Size([288]) 2.2691287994384766
torch.Size([1152, 288]) 74.26287078857422
torch.Size([1152]) 8.77828311920166
torch.Size([288, 1152]) 63.05274963378906
torch.Size([288]) 1.5091584920883179
torch.Size([288]) 12.185152053833008
torch.Size([288]) 8.095812797546387
torch.Size([288]) 11.048528671264648
torch.Size([288]) 2.020542621612549
_______________
torch.Size([864, 288]) 61.00320816040039
torch.Size([864]) 3.666269540786743
torch.Size([288, 288]) 19.89565658569336
torch.Size([288]) 2.0682854652404785
torch.Size([1152, 288]) 81.45234680175781
torch.Size([1152]) 6.818491458892822
torch.Size([288, 1152]) 69.92822265625
torch.Size([288]) 2.103447198867798
torch.Size([288]) 13.920809745788574
torch.Size([288]) 4.920451641082764
torch.Size([288]) 12.760061264038086
torch.Size([288]) 1.6274493932724
_______________
torch.Size([864, 288]) 61.021724700927734
torch.Size([86

In [25]:
# test_accu = Trainer_Class.test(test_data=test_data_loder_100,disable_pbar=False,enable_bf16=False)

In [26]:
print("\n\n\nTrain Accuracy and Losses")
for d in accu_and_loss[0]:
    print(d)
print("\nTest Accuracy and Losses")
for d in accu_and_loss[1]:
    print(d)




Train Accuracy and Losses
{'MLM_Loss': 5.805957701704912, 'MLM_Accu': 0.24385119467640384, 'Overall_Accu': 0.801719358642746}
{'MLM_Loss': 4.470974059644345, 'MLM_Accu': 0.3320003412347122, 'Overall_Accu': 0.8369373842772482}
{'MLM_Loss': 4.05716178799379, 'MLM_Accu': 0.3680979594112271, 'Overall_Accu': 0.8469939669887673}
{'MLM_Loss': 3.8305200698300212, 'MLM_Accu': 0.3902725517357908, 'Overall_Accu': 0.8537175848112089}
{'MLM_Loss': 3.7051917116968967, 'MLM_Accu': 0.40306295182419943, 'Overall_Accu': 0.8576404523714883}
{'MLM_Loss': 3.6609053998186676, 'MLM_Accu': 0.40717782450865575, 'Overall_Accu': 0.8579217049449036}
{'MLM_Loss': 3.564468107851888, 'MLM_Accu': 0.41769818753432186, 'Overall_Accu': 0.8619348974000604}

Test Accuracy and Losses
{'MLM_Loss': 4.703009593687811, 'MLM_Accu': 0.31708502619564355, 'Overall_Accu': 0.8378298796051127}
{'MLM_Loss': 4.113412705991381, 'MLM_Accu': 0.36576104374502877, 'Overall_Accu': 0.8528106150268722}
{'MLM_Loss': 3.816483093350572, 'MLM_A

In [27]:
with open(f"{model_name}_accuracy_and_loss.plk",'wb') as f:
    pickle.dump(obj=accu_and_loss,file=f)

In [28]:
print("\n\n\nSteps Acuracy Logs:")
for t in Trainer_Class.user_metrics_class.accu_over_steps():
    print(t)




Steps Acuracy Logs:
(699, 1, {'MLM_Loss': 8.391403280102631, 'MLM_Accu': 0.12125152051555407, 'Overall_Accu': 0.5346198791148228})
(1398, 1, {'MLM_Loss': 7.268434519760939, 'MLM_Accu': 0.1566092308413485, 'Overall_Accu': 0.7883051192129463})
(2097, 1, {'MLM_Loss': 7.094460927365676, 'MLM_Accu': 0.16699806192572642, 'Overall_Accu': 0.797648068917154})
(2796, 1, {'MLM_Loss': 6.869263609420932, 'MLM_Accu': 0.1825774400275146, 'Overall_Accu': 0.8016986422272747})
(3495, 1, {'MLM_Loss': 6.5657985800496155, 'MLM_Accu': 0.19826375709071148, 'Overall_Accu': 0.8047596242635885})
(4194, 1, {'MLM_Loss': 6.288635598402337, 'MLM_Accu': 0.21030843690655912, 'Overall_Accu': 0.8070454180496445})
(4893, 1, {'MLM_Loss': 6.044516606392256, 'MLM_Accu': 0.22178802109497786, 'Overall_Accu': 0.8087609216720033})
(5592, 1, {'MLM_Loss': 5.837597644380234, 'MLM_Accu': 0.23233682989477128, 'Overall_Accu': 0.8113090569028134})
(6291, 1, {'MLM_Loss': 5.663381250460602, 'MLM_Accu': 0.2441175819850275, 'Overall_A

In [29]:
print("\n\n\nLogs")
for t in Trainer_Class.user_metrics_class.checkpoint_metrices:
    print(t)




Logs
(699, 1, {'MLM_Loss': 8.391403280102631, 'MLM_Accu': 0.12125152051555407, 'Overall_Accu': 0.5346198791148228})
(1398, 1, {'MLM_Loss': 7.829918899931785, 'MLM_Accu': 0.1389303756784513, 'Overall_Accu': 0.6614624991638846})
(2097, 1, {'MLM_Loss': 7.584766242409748, 'MLM_Accu': 0.14828627109420967, 'Overall_Accu': 0.706857689081641})
(2796, 1, {'MLM_Loss': 7.405890584162544, 'MLM_Accu': 0.1568590633275359, 'Overall_Accu': 0.7305679273680494})
(3495, 1, {'MLM_Loss': 7.237872183339959, 'MLM_Accu': 0.16514000208017102, 'Overall_Accu': 0.7454062667471573})
(4194, 1, {'MLM_Loss': 7.079666085850355, 'MLM_Accu': 0.1726680745512357, 'Overall_Accu': 0.7556794586309051})
(4893, 1, {'MLM_Loss': 6.931787588784912, 'MLM_Accu': 0.1796852097717703, 'Overall_Accu': 0.7632625247796334})
(5592, 1, {'MLM_Loss': 6.795013845734328, 'MLM_Accu': 0.18626666228714542, 'Overall_Accu': 0.769268341295031})
(6291, 1, {'MLM_Loss': 6.669276890703913, 'MLM_Accu': 0.19269454225357677, 'Overall_Accu': 0.7742847078

In [30]:
Trainer_Class.user_metrics_class.save(model_name)

In [31]:
optim.optim.zero_grad(set_to_none=True)

In [32]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [33]:
gc.collect()

0